# 🔬 Unified Object Detection & Image Retrieval Joint Training (Tasks 1-4 Demo)
Notebook này hướng dẫn chi tiết cách chạy huấn luyện và đánh giá hệ thống hợp nhất sâu bệnh nông nghiệp dạng **Single-Stage** (sử dụng Retrieval Head và Triplet Loss) cho cả **4 Tasks** tuần tự trên Kaggle.

### ⚙️ Thiết kế mô hình mới:
1. **Retrieval Head**: Một nhánh tích chập được thêm vào đầu của mạng Detector, có nhiệm vụ trực tiếp xuất ra vector đặc trưng truy vấn 256 chiều.
2. **Triplet Loss**: Mô hình được tối ưu hoá trực tiếp bằng hàm mất mát bộ ba (Triplet Loss) để căn chỉnh không gian đặc trưng (embedding space) của sâu bệnh nông nghiệp.
3. **Huấn luyện đồng thời (Joint Training)**: Tổng loss bằng `loss_box + loss_cls + loss_retrieval`, tối ưu hoá toàn bộ mạng.

### ⚙️ Cấu hình huấn luyện:
- Số lượng Epochs: **1 epoch** cho mỗi Task
- Batch Size: **16**
- Huấn luyện liên tục (Incremental Learning): Task sau sẽ nạp checkpoint đã học của Task trước đó để tiếp tục huấn luyện.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (*Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Clone Repository từ GitHub
Tải mã nguồn cùng submodule `mmyolo` từ GitHub.

In [1]:
# Khai báo thông tin repo
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

# Tự động cập nhật file notebook đang chạy ở ngoài thư mục working_dir
# import glob, shutil
# for ipynb_path in glob.glob("/kaggle/working/*.ipynb"):
#     shutil.copy("New_retrival/retrival-img-latest.ipynb", ipynb_path)
#     print(f"-> Đã tự động cập nhật code mới cho notebook: {ipynb_path}")
#     print("   ==> QUAN TRỌNG: Hãy nhấn F5 (Reload lại trang trình duyệt) để áp dụng code mới trước khi chạy các ô tiếp theo!")


-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 1142, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 1142 (delta 122), reused 97 (delta 43), pack-reused 960 (from 1)
Receiving objects: 100% (1142/1142), 2.06 MiB | 9.30 MiB/s, done.
Resolving deltas: 100% (763/763), done.
/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 13.68 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi tương thích MMCV

In [2]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import os
import glob
import shutil

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

def clear_pycache(root_dir):
    if not os.path.exists(root_dir):
        return
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == "__pycache__":
                pycache_path = os.path.join(root, d)
                try:
                    shutil.rmtree(pycache_path)
                except Exception:
                    pass

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        pkg_dir = os.path.join(s_dir, pkg)
        patch_file(os.path.join(pkg_dir, "__init__.py"))
        clear_pycache(pkg_dir)

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)
        clear_pycache(os.path.dirname(init_file))

def verify_patch(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if "mmcv_maximum_version" in line:
                    print(f"  [Xác nhận] {file_path}: {line.strip()}")

for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        verify_patch(os.path.join(s_dir, pkg, "__init__.py"))
for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    verify_patch(init_file)

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo

mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

-> 1. Thiết lập phiên bản PyTorch & Torchvision...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 95.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 105.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 

## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng phụ trợ (Auxiliary Embeddings)
Tải pretrain weights của YOLO-World và tự động định vị dataset IP102 trên Kaggle để sinh các tệp đặc trưng cần thiết.

In [3]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Khởi tạo các thư mục
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights của YOLO-World làm nền tảng nếu cần
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

# 3. Định vị thư mục dataset IP102 trên Kaggle
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")

# 4. Thiết lập danh sách nhãn lớp chuẩn từ 0 đến 101
class_names = [str(i) for i in range(102)]

num_classes = len(class_names)
print(f"-> Tổng số lớp sâu bệnh: {num_classes}")

# 5. Lưu class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 6. Sinh class embeddings bằng CLIP
print("-> Đang trích xuất text embeddings bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

# 7. Sinh task_att_1_embeddings.pth
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 8. Sinh mowod_distribution_sim1.pth
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Sinh file đặc trưng phụ trợ thành công! ======")

-> Đang tải pretrained weights...
--2026-08-12 06:48:37--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 13.226.251.112, 13.226.251.66, 13.226.251.20, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.112|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%3B+filename%3D%22yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%22%3B&user_id=public&Expires=1786520917&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYjdhNzE2MjZhNGMyMDk5MDZhZGY1LzA5ZGFmYjczYjBkMTlkMjcwY2YyMGY3ZWVhYzZhNzg2MTMwM2E3NTMzMzJkNWRmOTkxNzc3MmJhMjNlNGE0N2RcXD9

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_ids                             | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.la

====== Sinh file đặc trưng phụ trợ thành công! ======


## 🛠️ Bước 3b: Khởi tạo mô hình và file cấu hình cục bộ
Tự động ghi mã nguồn của mô hình mới `our_head_retrieval.py` và cả **4 file cấu hình** cho Tasks 1, 2, 3, 4 trực tiếp vào ổ đĩa của Kaggle trước khi bắt đầu huấn luyện.

In [4]:
import os
os.makedirs('New_retrival', exist_ok=True)
print("-> Đang tạo file New_retrival/our_head_retrieval.py...")

-> Đang tạo file New_retrival/our_head_retrieval.py...


In [5]:
%%writefile New_retrival/our_head_retrieval.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Union, Sequence, Dict
from mmcv.cnn import ConvModule
from mmengine.model import BaseModule
from torch import Tensor
from mmdet.structures import SampleList
from mmdet.utils import OptConfigType, InstanceList, OptInstanceList
from mmdet.models.utils import multi_apply
from mmyolo.registry import MODELS
from yolo_world.models.dense_heads.our_head_new import OurHeadModule, OurHead
from mmyolo.models.utils import gt_instances_preprocess

class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        
    def forward(self, embeddings, labels):
        """
        embeddings: Tensor of shape (N, D) where N is the number of positive samples
        labels: Tensor of shape (N,) containing class IDs
        """
        if len(embeddings) < 3:
            return embeddings.new_tensor(0.0, requires_grad=True)
            
        # L2 distances between all positive embeddings
        dot_product = torch.matmul(embeddings, embeddings.t())
        square_norm = torch.diag(dot_product)
        distances = square_norm.unsqueeze(1) - 2.0 * dot_product + square_norm.unsqueeze(0)
        distances = torch.clamp(distances, min=0.0)
        
        # Mask for positive pairs (same label, different indices)
        label_equal = labels.unsqueeze(0) == labels.unsqueeze(1)
        indices_equal = torch.eye(labels.size(0), device=labels.device).bool()
        mask_pos = label_equal & ~indices_equal
        
        # Mask for negative pairs (different labels)
        mask_neg = ~label_equal
        
        triplet_loss = []
        N = len(embeddings)
        for i in range(N):
            pos_indices = torch.where(mask_pos[i])[0]
            neg_indices = torch.where(mask_neg[i])[0]
            if len(pos_indices) == 0 or len(neg_indices) == 0:
                continue
            
            d_ap = distances[i, pos_indices].unsqueeze(1) # (P, 1)
            d_an = distances[i, neg_indices].unsqueeze(0) # (1, N_neg)
            
            loss_temp = d_ap - d_an + self.margin
            loss_temp = torch.clamp(loss_temp, min=0.0)
            
            triplet_loss.append(loss_temp.mean())
            
        if len(triplet_loss) == 0:
            return embeddings.new_tensor(0.0, requires_grad=True)
            
        return torch.stack(triplet_loss).mean()

class AssignerWrapper(nn.Module):
    def __init__(self, original_assigner):
        super().__init__()
        self.original_assigner = original_assigner
        self.latest_result = None
        
    def forward(self, *args, **kwargs):
        res = self.original_assigner(*args, **kwargs)
        self.latest_result = res
        return res
        
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)
        
    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.original_assigner, name)


@MODELS.register_module()
class OurHeadRetrievalModule(OurHeadModule):
    def __init__(self, *args, retrieval_dim=256, **kwargs) -> None:
        self.retrieval_dim = retrieval_dim
        super().__init__(*args, **kwargs)
        
    def _init_layers(self) -> None:
        super()._init_layers()
        self.ret_preds = nn.ModuleList()
        cls_out_channels = max(self.in_channels[0], self.num_classes)
        for i in range(self.num_levels):
            self.ret_preds.append(
                nn.Sequential(
                    ConvModule(in_channels=self.in_channels[i],
                               out_channels=cls_out_channels,
                               kernel_size=3,
                               stride=1,
                               padding=1,
                               norm_cfg=self.norm_cfg,
                               act_cfg=self.act_cfg),
                    ConvModule(in_channels=cls_out_channels,
                               out_channels=cls_out_channels,
                               kernel_size=3,
                               stride=1,
                               padding=1,
                               norm_cfg=self.norm_cfg,
                               act_cfg=self.act_cfg),
                    nn.Conv2d(in_channels=cls_out_channels,
                              out_channels=self.retrieval_dim,
                              kernel_size=1)
                )
            )
            
    def forward(self, img_feats: Tuple[Tensor], txt_feats: Tensor) -> Tuple[List]:
        assert len(img_feats) == self.num_levels
        txt_feats = [txt_feats for _ in range(self.num_levels)]
        return multi_apply(self.forward_single, img_feats, txt_feats,
                           self.cls_preds, self.reg_preds, self.cls_contrasts, self.ret_preds)
                           
    def forward_single(self, img_feat: Tensor, txt_feat: Tensor,
                       cls_pred: nn.ModuleList, reg_pred: nn.ModuleList,
                       cls_contrast: nn.ModuleList, ret_pred: nn.ModuleList) -> Tuple:
        b, _, h, w = img_feat.shape
        cls_embed = cls_pred(img_feat)
        cls_logit = cls_contrast(cls_embed, txt_feat)
        bbox_dist_preds = reg_pred(img_feat)
        
        # Retrieval embedding projection and normalization
        ret_embed = ret_pred(img_feat)
        ret_embed = F.normalize(ret_embed, p=2, dim=1) # shape: (b, retrieval_dim, h, w)
        
        if self.reg_max > 1:
            bbox_dist_preds = bbox_dist_preds.reshape(
                [-1, 4, self.reg_max, h * w]).permute(0, 3, 1, 2)
            bbox_preds = bbox_dist_preds.softmax(3).matmul(
                self.proj.view([-1, 1])).squeeze(-1)
            bbox_preds = bbox_preds.transpose(1, 2).reshape(b, -1, h, w)
        else:
            bbox_preds = bbox_dist_preds
            
        if self.training:
            return cls_logit, bbox_preds, bbox_dist_preds, ret_embed
        else:
            return cls_logit, bbox_preds, ret_embed

@MODELS.register_module()
class OurHeadRetrieval(OurHead):
    def __init__(self, *args, loss_retrieval_weight=0.5, retrieval_dim=256, triplet_margin=0.3, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_retrieval_weight = loss_retrieval_weight
        self.retrieval_dim = retrieval_dim
        self.triplet_loss = TripletLoss(margin=triplet_margin)
        
        if hasattr(self, 'assigner') and self.assigner is not None:
            self.assigner = AssignerWrapper(self.assigner)
            
    def loss(self, img_feats: Tuple[Tensor], txt_feats: Tensor,
             batch_data_samples: Union[list, dict], fusion_att: bool=False) -> dict:
        # Dynamically wrap assigner if it wasn't available at initialization
        if hasattr(self, 'assigner') and self.assigner is not None:
            if not isinstance(self.assigner, AssignerWrapper):
                self.assigner = AssignerWrapper(self.assigner)
                
        # Forward pass of OurHeadRetrievalModule (returns 4 lists during training)
        outs = self(img_feats, txt_feats)
        cls_scores, bbox_preds, bbox_dist_preds, ret_embeds = outs
        
        # Pass detection outputs
        det_outs = (cls_scores, bbox_preds, bbox_dist_preds)
        
        if self.att_embeddings is None:
            loss_inputs = det_outs + (None, batch_data_samples['bboxes_labels'],
                                      batch_data_samples['img_metas'])
            losses = self.loss_by_feat(*loss_inputs)
        else:
            if fusion_att: 
                num_att = self.att_embeddings.shape[0]
                att_feats = txt_feats[:, -num_att: , :]
                txt_feats = txt_feats[:, :-num_att, :]
            else:
                att_feats = self.att_embeddings[None].repeat(txt_feats.shape[0], 1, 1)
            
            with torch.no_grad():
                att_outs = self(img_feats, att_feats)[0]
                
            loss_inputs = det_outs + (att_outs, batch_data_samples['bboxes_labels'],
                                      batch_data_samples['img_metas'])
            losses = self.loss_by_feat(*loss_inputs)
            
        # Retrieval Loss (Triplet Loss)
        if not hasattr(self, 'assigner') or self.assigner is None or self.assigner.latest_result is None:
            losses['loss_retrieval'] = ret_embeds[0].new_tensor(0.0, requires_grad=True)
            return losses
            
        assign_result = self.assigner.latest_result
        
        # Flatten ret_embeds to align with total anchors
        flatten_ret_embeds = []
        for ret_embed in ret_embeds:
            b, c, h, w = ret_embed.shape
            flat = ret_embed.view(b, c, -1).permute(0, 2, 1)
            flatten_ret_embeds.append(flat)
        flatten_ret_embeds = torch.cat(flatten_ret_embeds, dim=1) # (batch_size, total_anchors, retrieval_dim)
        
        pos_embeddings_list = []
        pos_labels_list = []
        
        # Handle dict format (e.g. BatchTaskAlignedAssigner) vs standard list/class format
        if isinstance(assign_result, dict):
            fg_mask_pre_prior = assign_result['fg_mask_pre_prior']
            assigned_bboxes = assign_result['assigned_bboxes']
            
            # Prepare gt_info using gt_instances_preprocess
            if isinstance(batch_data_samples, dict):
                gt_info = gt_instances_preprocess(batch_data_samples['bboxes_labels'], fg_mask_pre_prior.shape[0])
            else:
                batch_gt_instances = [sample.gt_instances for sample in batch_data_samples]
                gt_info = gt_instances_preprocess(batch_gt_instances, fg_mask_pre_prior.shape[0])
                
            for i in range(fg_mask_pre_prior.shape[0]):
                fg_mask = fg_mask_pre_prior[i]
                if not fg_mask.any():
                    continue
                    
                img_pos_embeds = flatten_ret_embeds[i][fg_mask]
                
                img_gt_boxes = gt_info[i, :, 1:].to(assigned_bboxes.device, dtype=assigned_bboxes.dtype)
                img_gt_labels = gt_info[i, :, 0].long().to(assigned_bboxes.device)
                
                valid_gt_mask = img_gt_boxes.sum(dim=-1) > 0
                img_gt_boxes = img_gt_boxes[valid_gt_mask]
                img_gt_labels = img_gt_labels[valid_gt_mask]
                
                img_pos_assigned_boxes = assigned_bboxes[i][fg_mask]
                
                # Match assigned boxes to ground truth boxes to get indices
                dists = torch.abs(img_pos_assigned_boxes.unsqueeze(1) - img_gt_boxes.unsqueeze(0)).sum(dim=-1)
                pos_gt_indices = torch.argmin(dists, dim=-1)
                
                img_pos_labels = img_gt_labels[pos_gt_indices]
                
                pos_embeddings_list.append(img_pos_embeds)
                pos_labels_list.append(img_pos_labels)
        else:
            # Original list/class format
            if isinstance(assign_result, list):
                gt_inds_list = [res.gt_inds for res in assign_result]
            else:
                gt_inds_list = assign_result.gt_inds
                if len(gt_inds_list.shape) == 2:
                    gt_inds_list = [gt_inds_list[i] for i in range(gt_inds_list.shape[0])]
                else:
                    gt_inds_list = [gt_inds_list]
                    
            # Prepare gt_info using gt_instances_preprocess
            if isinstance(batch_data_samples, dict):
                gt_info = gt_instances_preprocess(batch_data_samples['bboxes_labels'], len(gt_inds_list))
            else:
                batch_gt_instances = [sample.gt_instances for sample in batch_data_samples]
                gt_info = gt_instances_preprocess(batch_gt_instances, len(gt_inds_list))
                
            for i, gt_inds in enumerate(gt_inds_list):
                fg_mask = gt_inds > 0
                if not fg_mask.any():
                    continue
                    
                img_pos_embeds = flatten_ret_embeds[i][fg_mask]
                
                img_gt_labels = gt_info[i, :, 0].long().to(gt_inds.device)
                
                pos_gt_indices = gt_inds[fg_mask] - 1
                img_pos_labels = img_gt_labels[pos_gt_indices]
                
                pos_embeddings_list.append(img_pos_embeds)
                pos_labels_list.append(img_pos_labels)
                
        if len(pos_embeddings_list) > 0:
            all_pos_embeds = torch.cat(pos_embeddings_list, dim=0)
            all_pos_labels = torch.cat(pos_labels_list, dim=0)
            loss_triplet = self.triplet_loss(all_pos_embeds, all_pos_labels)
        else:
            loss_triplet = ret_embeds[0].new_tensor(0.0, requires_grad=True)
            
        losses['loss_retrieval'] = loss_triplet * self.loss_retrieval_weight
        return losses

    def predict(self,
                img_feats: Tuple[Tensor],
                txt_feats: Tensor,
                batch_data_samples: SampleList,
                rescale: bool = False, 
                fusion_att: bool = False) -> InstanceList:
        
        # Forward pass in evaluation/predict mode (returns 3 lists: cls_scores, bbox_preds, ret_embeds)
        x = self(img_feats, txt_feats)
        cls_scores, bbox_preds, ret_embeds = x
        
        # Pack only detection outputs
        det_outs = (cls_scores, bbox_preds)
        
        if self.att_embeddings.shape[0] != 25 * (self.num_classes):
            self.select_att()
            
        batch_img_metas = [
            data_samples.metainfo for data_samples in batch_data_samples
        ]
        
        if self.att_embeddings is None:
            predictions = self.predict_by_feat(*det_outs,
                                               batch_img_metas=batch_img_metas,
                                               rescale=rescale)
        else:
            if fusion_att: 
                num_att = self.att_embeddings.shape[0]
                att_feats = txt_feats[:, -num_att: , :]
                txt_feats = txt_feats[:, :-num_att, :]
            else:
                if self.attr_sel_for_known_only:
                    att_feats = self.all_atts[None].repeat(txt_feats.shape[0], 1, 1)
                else:
                    att_feats = self.att_embeddings[None].repeat(txt_feats.shape[0], 1, 1)
            
            det_outs = self.predict_unknown(det_outs, img_feats, att_feats)
            predictions = self.predict_by_feat(*det_outs,
                                               batch_img_metas=batch_img_metas,
                                               rescale=rescale)
            
        # Extract retrieval embeddings for the predicted bounding boxes
        import torchvision.ops as tv_ops
        for i, pred in enumerate(predictions):
            if len(pred) == 0:
                pred.features = pred.bboxes.new_zeros((0, self.retrieval_dim))
                continue
                
            bboxes = pred.bboxes
            img_meta = batch_img_metas[i]
            scale_factor = img_meta.get('scale_factor', (1.0, 1.0))
            if isinstance(scale_factor, float):
                scale_factor = (scale_factor, scale_factor)
                
            if rescale:
                w_scale, h_scale = scale_factor
                scaled_bboxes = bboxes.clone()
                scaled_bboxes[:, 0] /= w_scale
                scaled_bboxes[:, 2] /= w_scale
                scaled_bboxes[:, 1] /= h_scale
                scaled_bboxes[:, 3] /= h_scale
            else:
                scaled_bboxes = bboxes
                
            pooled_feats = []
            for level_idx, stride in enumerate([8, 16, 32]):
                feat = ret_embeds[level_idx][i:i+1] # (1, retrieval_dim, h_j, w_j)
                lvl_boxes = scaled_bboxes / stride
                rois = torch.cat([lvl_boxes.new_zeros(len(lvl_boxes), 1), lvl_boxes], dim=1)
                pooled = tv_ops.roi_align(feat, rois, output_size=(1, 1), spatial_scale=1.0, aligned=True)
                pooled_feats.append(pooled.view(len(lvl_boxes), -1))
                
            img_box_embeds = torch.stack(pooled_feats, dim=0).mean(dim=0)
            img_box_embeds = F.normalize(img_box_embeds, p=2, dim=1)
            pred.features = img_box_embeds
            
        return predictions



Overwriting New_retrival/our_head_retrieval.py


In [6]:
%%writefile New_retrival/ip102_t1_retrieval.py
_base_ = '../configs/open_world/mowod/custom/ip102_t1.py'

# Dynamically import our custom head from the local folder
custom_imports = dict(
    imports=['New_retrival.our_head_retrieval'],
    allow_failed_imports=False
)

# Replace box head types to use our new retrieval head and module
model = dict(
    bbox_head=dict(
        type='OurHeadRetrieval',
        retrieval_dim=256,
        loss_retrieval_weight=0.5,
        triplet_margin=0.3,
        head_module=dict(
            type='OurHeadRetrievalModule',
            retrieval_dim=256
        )
    )
)

# Training configurations: 1 epoch, batch size 16
max_epochs = 1
close_mosaic_epochs = 1
train_batch_size_per_gpu = 16

train_cfg = dict(max_epochs=max_epochs, val_interval=1)


Overwriting New_retrival/ip102_t1_retrieval.py


In [7]:
%%writefile New_retrival/ip102_t2_retrieval.py
_base_ = '../configs/open_world/mowod/custom/ip102_t2.py'

# Dynamically import our custom head from the local folder
custom_imports = dict(
    imports=['New_retrival.our_head_retrieval'],
    allow_failed_imports=False
)

# Replace box head types to use our new retrieval head and module
model = dict(
    bbox_head=dict(
        type='OurHeadRetrieval',
        retrieval_dim=256,
        loss_retrieval_weight=0.5,
        triplet_margin=0.3,
        head_module=dict(
            type='OurHeadRetrievalModule',
            retrieval_dim=256
        )
    )
)

# Training configurations: 1 epoch, batch size 16
max_epochs = 1
close_mosaic_epochs = 1
train_batch_size_per_gpu = 16

train_cfg = dict(max_epochs=max_epochs, val_interval=1)


Overwriting New_retrival/ip102_t2_retrieval.py


In [8]:
%%writefile New_retrival/ip102_t3_retrieval.py
_base_ = '../configs/open_world/mowod/custom/ip102_t3.py'

# Dynamically import our custom head from the local folder
custom_imports = dict(
    imports=['New_retrival.our_head_retrieval'],
    allow_failed_imports=False
)

# Replace box head types to use our new retrieval head and module
model = dict(
    bbox_head=dict(
        type='OurHeadRetrieval',
        retrieval_dim=256,
        loss_retrieval_weight=0.5,
        triplet_margin=0.3,
        head_module=dict(
            type='OurHeadRetrievalModule',
            retrieval_dim=256
        )
    )
)

# Training configurations: 1 epoch, batch size 16
max_epochs = 1
close_mosaic_epochs = 1
train_batch_size_per_gpu = 16

train_cfg = dict(max_epochs=max_epochs, val_interval=1)


Overwriting New_retrival/ip102_t3_retrieval.py


In [9]:
%%writefile New_retrival/ip102_t4_retrieval.py
_base_ = '../configs/open_world/mowod/custom/ip102_t4.py'

# Dynamically import our custom head from the local folder
custom_imports = dict(
    imports=['New_retrival.our_head_retrieval'],
    allow_failed_imports=False
)

# Replace box head types to use our new retrieval head and module
model = dict(
    bbox_head=dict(
        type='OurHeadRetrieval',
        retrieval_dim=256,
        loss_retrieval_weight=0.5,
        triplet_margin=0.3,
        head_module=dict(
            type='OurHeadRetrievalModule',
            retrieval_dim=256
        )
    )
)

# Training configurations: 1 epoch, batch size 16
max_epochs = 1
close_mosaic_epochs = 1
train_batch_size_per_gpu = 16

train_cfg = dict(max_epochs=max_epochs, val_interval=1)


Overwriting New_retrival/ip102_t4_retrieval.py


## Bước 4: Chạy huấn luyện Joint Training liên tục (Tasks 1-4)
Chạy huấn luyện liên tục. Để mô hình nhận diện được các file module nằm trong thư mục cục bộ `New_retrival`, lệnh huấn luyện được bổ sung biến môi trường **`PYTHONPATH=.`** nhằm khai báo thư mục gốc vào danh sách tìm kiếm package của Python.

In [10]:
print("-> 1. Đang huấn luyện Task 1 (1 epoch)... (Work-dir: New_retrival/work_dir/t1)")
!PYTHONPATH=. python third_party/mmyolo/tools/train.py New_retrival/ip102_t1_retrieval.py --work-dir New_retrival/work_dir/t1

print("\n-> 2. Đang huấn luyện Task 2 (1 epoch) kế thừa Task 1... (Work-dir: New_retrival/work_dir/t2)")
!PYTHONPATH=. python third_party/mmyolo/tools/train.py New_retrival/ip102_t2_retrieval.py --work-dir New_retrival/work_dir/t2 --cfg-options load_from=New_retrival/work_dir/t1/epoch_1.pth

print("\n-> 3. Đang huấn luyện Task 3 (1 epoch) kế thừa Task 2... (Work-dir: New_retrival/work_dir/t3)")
!PYTHONPATH=. python third_party/mmyolo/tools/train.py New_retrival/ip102_t3_retrieval.py --work-dir New_retrival/work_dir/t3 --cfg-options load_from=New_retrival/work_dir/t2/epoch_1.pth

print("\n-> 4. Đang huấn luyện Task 4 (1 epoch) kế thừa Task 3... (Work-dir: New_retrival/work_dir/t4)")
!PYTHONPATH=. python third_party/mmyolo/tools/train.py New_retrival/ip102_t4_retrieval.py --work-dir New_retrival/work_dir/t4 --cfg-options load_from=New_retrival/work_dir/t3/epoch_1.pth

print("====== Quá trình huấn luyện liên tục 4 Tasks hoàn tất! ======")

-> 1. Đang huấn luyện Task 1 (1 epoch)... (Work-dir: New_retrival/work_dir/t1)
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmdet/models/backbones/trident_resnet.py:244: SyntaxWarning: invalid escape sequence '\ '
  \ stage3(b2) /
/usr/local/lib/python3.12/dist-packages/mmdet/models/dense_heads/free_anchor_retina_head.py:290: SyntaxWarning: invalid escape sequence '\i'
  :math:`FL((1 - P_{a_{j} \in A_{+}}) * (1 - P_{j}^{bg}))`.
08/12 06:50:29 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when runni

## Bước 5: Chạy đánh giá Image Retrieval cho cả 4 Tasks sau khi huấn luyện
Đo lường độ chính xác (Recall@1/5/10, AUROC, FPR@95) tuần tự cho cả 4 tasks bằng các checkpoint tương ứng vừa học.

In [11]:
import os
from IPython.display import Markdown, display

tasks_eval = [
    {"id": 1, "config": "New_retrival/ip102_t1_retrieval.py", "checkpoint": "New_retrival/work_dir/t1/epoch_1.pth"},
    {"id": 2, "config": "New_retrival/ip102_t2_retrieval.py", "checkpoint": "New_retrival/work_dir/t2/epoch_1.pth"},
    {"id": 3, "config": "New_retrival/ip102_t3_retrieval.py", "checkpoint": "New_retrival/work_dir/t3/epoch_1.pth"},
    {"id": 4, "config": "New_retrival/ip102_t4_retrieval.py", "checkpoint": "New_retrival/work_dir/t4/epoch_1.pth"}
]

for t in tasks_eval:
    task_id = t["id"]
    config_path = t["config"]
    checkpoint_path = t["checkpoint"]
    report_file = f'New_retrival/report_task{task_id}_retrieval.md'
    
    print("\n" + "="*60)
    print(f"   ĐANG CHẠY ĐÁNH GIÁ TRUY VẤN ẢNH CHO TASK {task_id}   ")
    print("="*60)
    
    if not os.path.exists(checkpoint_path):
        print(f"⚠️ Không tìm thấy checkpoint tại: {checkpoint_path}")
        continue
        
    !PYTHONPATH=. python evaluate_retrieval.py \
        --config "{config_path}" \
        --checkpoint "{checkpoint_path}" \
        --dataset-root "{dataset_root}" \
        --query-split val \
        --gallery-split test \
        --query-cache "New_retrival/query_cache_task{task_id}_new_model.pkl" \
        --gallery-cache "New_retrival/gallery_cache_task{task_id}_new_model.pkl" \
        --output-report "{report_file}" \
        --device cuda:0 \
        --detector-retrieval
        
    if os.path.exists(report_file):
        print(f"\n🔍 KẾT QUẢ ĐÁNH GIÁ TASK {task_id} (Đọc từ {report_file}):")
        display(Markdown(filename=report_file))
    else:
        print(f"⚠️ Không tìm thấy báo cáo kết quả '{report_file}'.")


   ĐANG CHẠY ĐÁNH GIÁ TRUY VẤN ẢNH CHO TASK 1   
⚠️ Không tìm thấy checkpoint tại: New_retrival/work_dir/t1/epoch_1.pth

   ĐANG CHẠY ĐÁNH GIÁ TRUY VẤN ẢNH CHO TASK 2   
⚠️ Không tìm thấy checkpoint tại: New_retrival/work_dir/t2/epoch_1.pth

   ĐANG CHẠY ĐÁNH GIÁ TRUY VẤN ẢNH CHO TASK 3   
⚠️ Không tìm thấy checkpoint tại: New_retrival/work_dir/t3/epoch_1.pth

   ĐANG CHẠY ĐÁNH GIÁ TRUY VẤN ẢNH CHO TASK 4   
⚠️ Không tìm thấy checkpoint tại: New_retrival/work_dir/t4/epoch_1.pth
